# Testing file 
### where we evaluate Zhang's models using the test set

## Preliminaries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.optimizers import Adam
from tensorflow.data import Dataset


from util.load_data import load_data
from util.evaluation import *
from models.zhang.models import FairLogisticRegression
from models.zhang.learning import train_loop as zhang_train

/Users/lffpl/Projects/falsb/env/falsb/lib/python3.11/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:
batch_size = 64
epochs = 100
lr = 0.001

In [3]:
cv_seeds = [13, 29, 42, 55, 73]

## Load data

In [4]:
data_name = 'balanced-stroke'

In [5]:
x, y, a = load_data(data_name)
raw_data = (x, y, a)

In [6]:
xdim = x.shape[1]
ydim = y.shape[1]
adim = a.shape[1]
zdim = 8

In [7]:
print(xdim, ydim, adim, zdim)

17 1 1 8


In [8]:
print(len(x))

9398


## Result file

In [9]:
header = "model_name", "cv_seed", "clas_acc", "dp", "deqodds", "deqopp", "trade_dp", "trade_deqodds", "trade_deqopp", "TN_a0", "FP_a0", "FN_a0", "TP_a0", "TN_a1", "FP_a1", "FN_a1", "TP_a1"
results = []

## Testing loop
#### Each model is evalueted 5 times
#### In the end of each iteration we save the result

### Zhang for DP

In [10]:
fairdef = 'DemPar'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)

    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4DP', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc


2025-12-30 16:21:49.371075: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 1 | 0.6354970932006836 | 0.8468934297561646 | 0.5419730392156863 | 0.4730392156862745


2025-12-30 16:21:51.611572: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 2 | 0.6007452011108398 | 0.8235824108123779 | 0.6836703431372549 | 0.4730392156862745
> 3 | 0.5684027671813965 | 0.7940462231636047 | 0.7454044117647058 | 0.4730392156862745


...

> 64 | 0.2562555968761444 | 0.6727490425109863 | 0.9021139705882353 | 0.6173406862745098
> 65 | 0.255725622177124 | 0.6732051968574524 | 0.9019607843137255 | 0.6167279411764706
> 66 | 0.25581198930740356 | 0.6737961769104004 | 0.9022671568627451 | 0.6158088235294118
> 67 | 0.25511905550956726 | 0.6741130948066711 | 0.9022671568627451 | 0.6153492647058824
> 68 | 0.2552168071269989 | 0.6747474074363708 | 0.9022671568627451 | 0.6144301470588235
> 69 | 0.2553899884223938 | 0.6755006909370422 | 0.9016544117647058 | 0.6136642156862745
> 70 | 0.2538456916809082 | 0.6752654314041138 | 0.9019607843137255 | 0.6128982843137255
> 71 | 0.2547931969165802 | 0.6764073371887207 | 0.9018075980392157 | 0.6122855392156863
> 72 | 0.25319576263427734 | 0.6761685609817505 | 0.9018075980392157 | 0.6119791666666666
> 73 | 0.25328442454338074 | 0.6768680810928345 | 0.9013480392156863 | 0.6116727941176471
> 74 | 0.2537846565246582 | 0.6777475476264954 | 0.9021139705882353 | 0.6122855392156863
> 75 | 0.252918720

2025-12-30 16:26:30.741407: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 27 | 0.30708345770835876 | 0.6671475172042847 | 0.8929227941176471 | 0.6478247549019608
> 28 | 0.30594223737716675 | 0.6679455041885376 | 0.8927696078431373 | 0.6458333333333334
> 29 | 0.3051702678203583 | 0.6689035296440125 | 0.8932291666666666 | 0.6441482843137255
> 30 | 0.3052360713481903 | 0.6694345474243164 | 0.8926164215686274 | 0.6441482843137255
> 31 | 0.303311288356781 | 0.6688011288642883 | 0.8935355392156863 | 0.6439950980392157
> 32 | 0.30346792936325073 | 0.6703767776489258 | 0.8943014705882353 | 0.6432291666666666
> 33 | 0.30378633737564087 | 0.6711791753768921 | 0.8944546568627451 | 0.6427696078431373
> 34 | 0.3044474720954895 | 0.671769917011261 | 0.8949142156862745 | 0.6421568627450981
> 35 | 0.3027189373970032 | 0.6714400053024292 | 0.8952205882352942 | 0.6415441176470589
> 36 | 0.30345219373703003 | 0.6722921133041382 | 0.8956801470588235 | 0.6407781862745098
> 37 | 0.30209898948669434 | 0.6726688146591187 | 0.8955269607843137 | 0.6393995098039216
> 38 | 0.30300623

2025-12-30 16:31:14.293722: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 54 | 0.38198262453079224 | 0.6458349823951721 | 0.9073223039215687 | 0.6266850490196079
> 55 | 0.38545212149620056 | 0.6460216045379639 | 0.90625 | 0.6263786764705882
> 56 | 0.3827718496322632 | 0.6476103067398071 | 0.9079350490196079 | 0.625765931372549
> 57 | 0.38614505529403687 | 0.6477696299552917 | 0.90625 | 0.625765931372549
> 58 | 0.3853653073310852 | 0.6490942239761353 | 0.9071691176470589 | 0.6239276960784313
> 59 | 0.38680464029312134 | 0.6496827602386475 | 0.907015931372549 | 0.6233149509803921
> 60 | 0.38600292801856995 | 0.6508490443229675 | 0.9068627450980392 | 0.6227022058823529
> 61 | 0.38741958141326904 | 0.6514350175857544 | 0.9082414215686274 | 0.6213235294117647
> 62 | 0.3865707516670227 | 0.6525511145591736 | 0.9079350490196079 | 0.6202512254901961
> 63 | 0.38612598180770874 | 0.6538056135177612 | 0.9094669117647058 | 0.6200980392156863
> 64 | 0.38934698700904846 | 0.6538771986961365 | 0.9087009803921569 | 0.6197916666666666
> 65 | 0.38873839378356934 | 0.6550796

### Zhang for Eq Odds

In [11]:
fairdef = 'EqOdds'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)
    
    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4EqOdds', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc
> 1 | 0.6377251148223877 | 0.8343243598937988 | 0.5386029411764706 | 0.4730392156862745
> 2 | 0.6010777950286865 | 0.7900764346122742 | 0.6714154411764706 | 0.4730392156862745
> 3 | 0.5677098035812378 | 0.7483855485916138 | 0.7380514705882353 | 0.4730392156862745
> 4 | 0.5365406274795532 | 0.7046067714691162 | 0.7864583333333334 | 0.4730392156862745
> 5 | 0.5114684104919434 | 0.6688874363899231 | 0.8114276960784313 | 0.5513174019607843
> 6 | 0.4908665716648102 | 0.6422791481018066 | 0.8278186274509803 | 0.6683517156862745


2025-12-30 17:00:22.650396: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 7 | 0.47071006894111633 | 0.6200633645057678 | 0.8374693627450981 | 0.6853553921568627
> 8 | 0.4541635513305664 | 0.6074878573417664 | 0.8488051470588235 | 0.6887254901960784
> 9 | 0.43614664673805237 | 0.6056407690048218 | 0.8515625 | 0.6864276960784313
> 10 | 0.4204082787036896 | 0.6054474115371704 | 0.8589154411764706 | 0.6852022058823529
> 11 | 0.4125407338142395 | 0.6047049760818481 | 0.860140931372549 | 0.6807598039215687
> 12 | 0.3990587890148163 | 0.6095602512359619 | 0.8639705882352942 | 0.6798406862745098
> 13 | 0.38807475566864014 | 0.613777220249176 | 0.8656556372549019 | 0.6764705882352942
> 14 | 0.37802714109420776 | 0.6181458234786987 | 0.8719362745098039 | 0.6757046568627451
> 15 | 0.3711732029914856 | 0.6208271980285645 | 0.8748468137254902 | 0.6704963235294118
> 16 | 0.36195844411849976 | 0.6252883672714233 | 0.8786764705882353 | 0.6703431372549019
> 17 | 0.35435640811920166 | 0.6287283897399902 | 0.8809742647058824 | 0.6668198529411765
> 18 | 0.3506484031677246 | 0

### Zhang for Eq Opp

In [12]:
fairdef = 'EqOpp'

for cv_seed in cv_seeds:
    x_train, x_test, y_train, y_test, a_train, a_test = train_test_split(
        x, y, a, test_size=0.3, random_state=cv_seed)

    train_data = Dataset.from_tensor_slices((x_train, y_train, a_train))
    train_data = train_data.batch(batch_size, drop_remainder=True)

    test_data = Dataset.from_tensor_slices((x_test, y_test, a_test))
    test_data = test_data.batch(batch_size, drop_remainder=True)

    # train below

    opt = Adam(learning_rate=lr)
    
    model = FairLogisticRegression(xdim, ydim, adim, batch_size, fairdef)
    zhang_train(model, raw_data, train_data, epochs, opt)

    Y, A, Y_hat, A_hat = fair_evaluation(model, test_data)
    clas_acc, dp, deqodds, deqopp, confusion_matrix, metrics_a0, metrics_a1 = compute_metrics(Y, A, Y_hat, A_hat, adim)

    fair_metrics = (dp, deqodds, deqopp)
    tradeoff = []
    for fair_metric in fair_metrics:
        tradeoff.append(compute_tradeoff(clas_acc, fair_metric))

    result = ['Zhang4EqOpp', cv_seed, clas_acc, dp, deqodds, deqopp, tradeoff[0], tradeoff[1], tradeoff[2]] + metrics_a0 + metrics_a1

    results.append(result)

    del(opt)

> Epoch | Class Loss | Adv Loss | Class Acc | Adv Acc
> 1 | 0.6353387236595154 | 0.483664333820343 | 0.5428921568627451 | 0.4730392156862745
> 2 | 0.6020872592926025 | 0.44653594493865967 | 0.6796875 | 0.4730392156862745
> 3 | 0.5675978660583496 | 0.41458389163017273 | 0.7469362745098039 | 0.4730392156862745
> 4 | 0.5422763824462891 | 0.39006876945495605 | 0.7826286764705882 | 0.47349877450980393
> 5 | 0.5145970582962036 | 0.358397901058197 | 0.8079044117647058 | 0.5709252450980392
> 6 | 0.4905323088169098 | 0.34494492411613464 | 0.8278186274509803 | 0.6527267156862745
> 7 | 0.4741412401199341 | 0.32641464471817017 | 0.8439031862745098 | 0.6608455882352942
> 8 | 0.45617541670799255 | 0.3204612731933594 | 0.8472732843137255 | 0.6585477941176471
> 9 | 0.4367898106575012 | 0.32165172696113586 | 0.8543198529411765 | 0.6605392156862745
> 10 | 0.42159947752952576 | 0.32384979724884033 | 0.8638174019607843 | 0.6582414215686274
> 11 | 0.4082358181476593 | 0.33162838220596313 | 0.86688112745098

2025-12-30 17:24:45.932148: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


> 14 | 0.37373703718185425 | 0.34018194675445557 | 0.8736213235294118 | 0.6530330882352942
> 15 | 0.364778995513916 | 0.34289687871932983 | 0.8768382352941176 | 0.6527267156862745
> 16 | 0.3588868975639343 | 0.34280312061309814 | 0.8799019607843137 | 0.6499693627450981
> 17 | 0.35302960872650146 | 0.34298714995384216 | 0.8821997549019608 | 0.6516544117647058
> 18 | 0.34609657526016235 | 0.344924658536911 | 0.8828125 | 0.6493566176470589
> 19 | 0.33896005153656006 | 0.3469037413597107 | 0.8867953431372549 | 0.6490502450980392
> 20 | 0.3341578245162964 | 0.34646597504615784 | 0.8883272058823529 | 0.6462928921568627
> 21 | 0.3272622227668762 | 0.3487713634967804 | 0.8895526960784313 | 0.6450674019607843
> 22 | 0.3237811326980591 | 0.3481045663356781 | 0.8909313725490197 | 0.6439950980392157
> 23 | 0.3192852735519409 | 0.34865280985832214 | 0.8918504901960784 | 0.6438419117647058
> 24 | 0.31528839468955994 | 0.34912627935409546 | 0.8929227941176471 | 0.6407781862745098
> 25 | 0.31167736649

## Saving into DF then CSV

In [13]:
result_df = pd.DataFrame(results, columns=header)
result_df

,model_name,cv_seed,clas_acc,dp,deqodds,deqopp,trade_dp,trade_deqodds,trade_deqopp,TN_a0,FP_a0,FN_a0,TP_a0,TN_a1,FP_a1,FN_a1,TP_a1
0,Zhang4DP,13,0.901278,0.758741,0.934602,0.895377,0.823890,0.917638,0.898318,816.0,27.0,116.0,410.0,534.0,33.0,102.0,778.0
1,Zhang4DP,29,0.887784,0.751170,0.908246,0.870082,0.813784,0.897898,0.878844,734.0,64.0,108.0,424.0,505.0,78.0,66.0,837.0
2,Zhang4DP,42,0.890980,0.746413,0.909658,0.867475,0.812315,0.900222,0.879070,777.0,56.0,114.0,420.0,506.0,66.0,71.0,806.0
3,Zhang4DP,55,0.887784,0.758276,0.919558,0.901057,0.817935,0.903392,0.894371,736.0,71.0,84.0,400.0,533.0,94.0,67.0,831.0
4,Zhang4DP,73,0.892045,0.753049,0.917232,0.888626,0.816675,0.904464,0.890332,752.0,47.0,105.0,408.0,526.0,67.0,85.0,826.0
5,Zhang4EqOdds,13,0.901634,0.759905,0.936002,0.896928,0.824724,0.918496,0.899275,809.0,34.0,111.0,415.0,530.0,37.0,95.0,785.0
6,Zhang4EqOdds,29,0.889205,0.752437,0.910211,0.876056,0.815124,0.899585,0.882581,737.0,61.0,106.0,426.0,506.0,77.0,68.0,835.0
7,Zhang4EqOdds,42,0.888494,0.757690,0.920411,0.881649,0.817896,0.904171,0.885058,784.0,49.0,118.0,416.0,515.0,57.0,90.0,787.0
8,Zhang4EqOdds,55,0.888494,0.751965,0.912925,0.890257,0.814548,0.900544,0.889375,747.0,60.0,93.0,391.0,540.0,87.0,74.0,824.0
9,Zhang4EqOdds,73,0.897727,0.762519,0.929691,0.895704,0.824618,0.913429,0.896714,762.0,37.0,107.0,406.0,544.0,49.0,95.0,816.0


In [14]:
result_df.to_csv(f'{data_name}-result/zhang-{epochs}.csv')